In [ ]:
# Cell 1: Setup
"""
# ArcticDEM Strip Difference Analysis
Compute and visualize elevation differences between pairs of DEM strips.
Works directly with coregistered GeoTIFF files - no preprocessing needed.
"""

import sys
sys.path.append('..')
import importlib
from config import OUTPUT_DIR, ARCHIVE_DIR
import diff_analysis_strips
importlib.reload(diff_analysis_strips)
from diff_analysis_strips import run_dem_difference, compute_dem_difference_from_geotiffs

print("✓ Modules loaded")

In [ ]:
# Cell 2: Configuration
"""
## Configuration
Edit these parameters for your analysis.
"""

# Optional: suffix pattern for coregistered files
# Example: '_mosaic_v_0-0_dh_0-0000_[\'vertical_offset_mean\', \'nuthkaab\', \'deramp\']_coregistered'
COREG_SUFFIX = "_mosaic_v_0-0_dh_0-0000_['vertical_offset_mean', 'nuthkaab', 'deramp']_coregistered"  # None = search for any coregistered file

# Tile identifier
TILE = "31_38_1_1"  # <-- EDIT THIS

# Optional shapefile for overlay (e.g., lake outlines)
petermann = "/home/moralpom/luna/CPOM/moralpom/globe/data/anna_lake_shapefile/20260426-20260427-20260508-20260509-lake.shp"
SHP_PATH = petermann  # <-- EDIT THIS or set to None

# Define strip pairs to compare (recent, old)
# Can use partial names - the function will find matching files
STRIP_PAIRS = [
    (
        "WV02_20180326_103001007AC58F00",  # Newer strip
        "WV02_20180308_103001007A136300",  # Older strip
    ),
    # Add more pairs as needed:
    # ("partial_name_a", "partial_name_b"),
]

print(f"Archive directory: {ARCHIVE_DIR}")
print(f"Coreg suffix: {COREG_SUFFIX}")
print(f"Tile: {TILE}")
print(f"Number of pairs: {len(STRIP_PAIRS)}")
print(f"Shapefile: {SHP_PATH}")

In [ ]:
# Cell 3: Run Analysis (updated)
"""
## Run DEM Difference Analysis

**Two processing modes:**

1. **Zarr (fast)**: Uses pre-computed zarr files - takes seconds per pair
2. **GeoTIFF (slower)**: Reads original GeoTIFFs with configurable downsampling

The function tries zarr first, then falls back to GeoTIFFs.
"""

# === CONFIGURE YOUR DATA SOURCES ===

# Option A: Zarr files (fast - seconds per pair)
ZARR_DIR = "/path/to/filled_arrays/zarr/"  # <-- EDIT or set to None
ARCHIVE_DIR = None  # Not needed if zarr works

# Option B: GeoTIFFs with downsampling (slower but still reasonable)
# ZARR_DIR = None
# ARCHIVE_DIR = "/path/to/arcticdem/archive/"  # <-- EDIT THIS
# DOWNSAMPLE = 10  # 10x downsampling = 20m resolution (from 2m)

# Option C: Both - zarr preferred, GeoTIFF fallback
# ZARR_DIR = "/path/to/zarr/"
# ARCHIVE_DIR = "/path/to/archive/"
# DOWNSAMPLE = 10

# For this run:
ZARR_DIR = "/home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/temp/filled_arrays/31_38_1_1/zarr/"
ARCHIVE_DIR = None
DOWNSAMPLE = None  # Not used when zarr is primary

results = process_strip_pairs(
    strip_pairs=STRIP_PAIRS,
    zarr_dir=ZARR_DIR,
    archive_dir=ARCHIVE_DIR,
    coreg_suffix=COREG_SUFFIX,
    output_dir=OUTPUT_DIR,
    prefer_zarr=True,              # Try zarr first
    downsample_factor=DOWNSAMPLE,  # For GeoTIFF fallback
    max_pixels=2000,               # Auto-downsample target
    cmap="RdBu_r",
    auto_crop=True,
    shp_path=SHP_PATH,
)

print("\n✓ Analysis complete!")

In [ ]:
# Cell 3: Run Analysis
"""
## Run DEM Difference Analysis
The function will:
1. Find matching GeoTIFF files for each strip
2. Calculate the overlapping region
3. Reproject/resample to common grid if needed
4. Compute elevation differences
5. Generate difference maps
"""

results = run_dem_difference(
    strip_pairs=STRIP_PAIRS,
    archive_dir=ARCHIVE_DIR,
    output_dir=OUTPUT_DIR,
    coreg_suffix=COREG_SUFFIX,
    tile=TILE,
    shp_path=SHP_PATH,
    cmap="RdBu_r",
    auto_crop=True,
    crop_threshold=0.5,
    max_pixels=2000,
)

print("\n✓ Analysis complete!")

In [ ]:
# Cell 4: View Results
"""
## Results Summary
"""

from IPython.display import Image, display

for (strip_a, strip_b), data in results.items():
    print(f"\n{'='*60}")
    print(f"Pair: {strip_a[:50]}...")
    print(f"      {strip_b[:50]}...")
    print(f"{'='*60}")
    
    stats = data['stats']
    print(f"Overlap shape: {stats['shape']}")
    print(f"Valid pixels: {stats['valid_pixels']:,}")
    print(f"Mean difference: {stats['mean']:.2f} m")
    print(f"Std difference: {stats['std']:.2f} m")
    print(f"Min/Max: {stats['min']:.2f} / {stats['max']:.2f} m")
    
    if data['plot_path']:
        print(f"\nPlot: {data['plot_path']}")
        display(Image(filename=data['plot_path']))